##3er modelo de regresion : Gradient Boosting Regressor

In [ ]:
# IMPORTACIONES
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

# MODELO BASE
model = GradientBoostingRegressor(
    random_state=42
)

# PIPELINE
clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

# GRID DE HIPERPARÁMETROS
param_grid = {
    "model__n_estimators": [100, 150],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [3, 5],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

# GRIDSEARCHCV
grid_search = GridSearchCV(
    estimator=clf,
    param_grid=param_grid,
    cv=3,
    scoring="r2",
    verbose=2,
    n_jobs=-1
)

# ENTRENAMIENTO
grid_search.fit(X_train, y_train)

# MEJORES PARÁMETROS
print("Mejores parámetros encontrados:", grid_search.best_params_)
print("Mejor R2 score en validación cruzada:", grid_search.best_score_)

# MEJOR MODELO
best_model = grid_search.best_estimator_

# PREDICCIONES
y_pred = best_model.predict(X_test)

# EVALUACIÓN
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE del mejor modelo (X_test):", mae)
print("R2 del mejor modelo (X_test):", r2)

##Segundo modelo de clasificacion : RandomForestClassifier

In [ ]:
# IMPORTACIONES
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

# VARIABLES
X = df[["Year", "Platform", "Genre", "Publisher"]]
y = df["Exitoso"]

# SPLIT DE DATOS
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# MODELO
model = RandomForestClassifier(
    random_state=42
)

# PIPELINE (Preprocessor + SMOTE + Modelo)
clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", model)
])

# GRIDSEARCH
param_grid = {
    "model__n_estimators": [50, 100, 150],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__criterion": ["gini", "entropy"]
}

# GRIDSEARCHCV
grid_search = GridSearchCV(
    estimator=clf,
    param_grid=param_grid,
    cv=3,
    scoring="f1",
    verbose=2,
    n_jobs=-1
)

# ENTRENAMIENTO
grid_search.fit(X_train, y_train)

# MEJOR MODELO
best_model = grid_search.best_estimator_

# PREDICCIONES
y_pred = best_model.predict(X_test)

# MÉTRICAS
print("Mejores parámetros:", grid_search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

print("\nReporte de clasificación:\n")
print(classification_report(y_test, y_pred))

## 2do modelo no supervisado DBSCAN(CLUSTERING ESPACIAL BASADO EN DENSIDAD DE APLICACIONES CON RUIDO)

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score
import numpy as np

# VARIABLES (sin target)
X_unsup = df[
    [
        "Year",
        "NA_Sales",
        "EU_Sales",
        "JP_Sales",
        "Other_Sales",
        "Platform",
        "Genre",
        "Publisher"
    ]
]

# DEFINIR COLUMNAS
num_cols = [
    "Year",
    "NA_Sales",
    "EU_Sales",
    "JP_Sales",
    "Other_Sales"
]

cat_cols = [
    "Platform",
    "Genre",
    "Publisher"
]

# PREPROCESSOR
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

# PIPELINE DBSCAN 
dbscan_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DBSCAN(
        eps=1.2,          
        min_samples=10    
    ))
])

# ENTRENAMIENTO
clusters_dbscan = dbscan_pipeline.fit_predict(X_unsup)

# guardar clusters
df["Cluster_DBSCAN"] = clusters_dbscan

# EVALUACIÓN
unique_labels = np.unique(clusters_dbscan)

# excluir ruido
clustered_idx = clusters_dbscan != -1

# cantidad de clusters reales
n_clusters = len(set(clusters_dbscan)) - (1 if -1 in clusters_dbscan else 0)

print("Número de clusters encontrados:", n_clusters)
print("Ruido (-1):", np.sum(clusters_dbscan == -1))

# calcular silhouette solo si hay >1 cluster válido
if n_clusters > 1:
    X_processed = preprocessor.fit_transform(X_unsup)

    sil_score = silhouette_score(
        X_processed[clustered_idx],
        clusters_dbscan[clustered_idx]
    )

    print("Silhouette Score:", sil_score)
else:
    print("No se pudo calcular Silhouette Score.")

# RESULTADOS
print("\nCantidad por cluster:")
print(df["Cluster_DBSCAN"].value_counts())

# EJEMPLOS
print("\nPrimeros registros:")
print(
    df[
        [
            "Name",
            "Platform",
            "Genre",
            "Global_Sales",
            "Cluster_DBSCAN"
        ]
    ].head(10)
)

# VER SOLO DATOS NO RUIDO
print("\nRegistros agrupados (sin ruido):")
print(
    df[df["Cluster_DBSCAN"] != -1][
        [
            "Name",
            "Platform",
            "Genre",
            "Global_Sales",
            "Cluster_DBSCAN"
        ]
    ].head(20)
)